**Grok signature:** with `wd=2.0`, `acc_full` saturates early while `symbolic_reliance` jumps
**late**; the `wd=0` control stays parametric (low reliance).

> Runtime → **T4 GPU**. ~20k steps ≈ 20–30 min — this validates the pipeline + early dynamics;
> a full transition may need the faithful long run (He et al. use 200k).

In [ ]:
![ -d algebra-grok ] || git clone -q https://github.com/tohuya6/algebra-grok.git
%cd algebra-grok/alg-grok
!pip -q install sympy
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(cpu)')

In [ ]:
# Two runs at fixed_p=1: weight decay on (grok candidate) vs off (parametric control).
# Uses `!python` (not subprocess) so progress + any tracebacks stream inline in Colab.
!python experiments/train_fixed_p.py --name fsg-fp1-wd2 --device cuda --fixed_p 1.0 --weight_decay 2.0 --lr 1.5e-4 --lr_warmup_steps 1000 --n_steps 20000 --evaluation_steps 100 --d_model 256 --n_layers 4 --n_heads 4 --num_symbols 12 --max_order 6 --k_shots 32 --batch_size 256
!python experiments/train_fixed_p.py --name fsg-fp1-wd0 --device cuda --fixed_p 1.0 --weight_decay 0.0 --lr 1.5e-4 --lr_warmup_steps 1000 --n_steps 20000 --evaluation_steps 100 --d_model 256 --n_layers 4 --n_heads 4 --num_symbols 12 --max_order 6 --k_shots 32 --batch_size 256

In [ ]:
# Grokking curves (log-x): memorization (acc_full) vs generalization (symbolic_reliance).
import json
import matplotlib.pyplot as plt

RUNS = {'wd=2.0 (grok?)': 'fsg-fp1-wd2', 'wd=0 (control)': 'fsg-fp1-wd0'}
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for label, name in RUNS.items():
    h = [r for r in json.load(open(f'outputs/{name}/metrics.json')) if r['step'] > 0]
    step = [r['step'] for r in h]
    ax[0].plot(step, [r['acc_full'] for r in h], label=label)
    ax[1].plot(step, [r['symbolic_reliance'] for r in h], label=label)
    loss = [(r['step'], r['train_loss']) for r in h if r['train_loss'] is not None]
    ax[2].plot([s for s, _ in loss], [l for _, l in loss], label=label)

for a, title, ylabel in zip(ax,
        ['acc_full (memorization)', 'symbolic_reliance (generalization)', 'train_loss'],
        ['acc_full', 'reliance', 'loss']):
    a.set(title=title, xlabel='step', ylabel=ylabel)
    a.set_xscale('log'); a.legend(); a.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()